# Tokyo 2020 Olympics — Tweets Frequency Analysis

Analyzing tweet frequency, sentiment patterns, and engagement metrics from social media conversations during the Tokyo 2020 (held in 2021) Olympic Games.

## Project Overview

The Tokyo 2020 Olympics were uniquely held in 2021 due to COVID-19, making them one of the most discussed sporting events on social media. This analysis examines tweet data to understand:

- How tweet volume fluctuated during the games
- Which topics and hashtags dominated conversations
- Temporal patterns in social media engagement
- Language and geographic distribution of tweets

## Learning Objectives

1. Work with text/social media data in a structured format
2. Perform time-series frequency analysis on event data
3. Extract and analyze hashtags and mentions from text
4. Create temporal visualizations (tweet volume over time)
5. Apply basic text processing techniques
6. Identify trends and patterns in social media data

## Business / Research Problem

**Core question:** What were the temporal patterns and content themes in Twitter conversations during the Tokyo 2020 Olympics?

Understanding social media patterns around major events helps marketers time campaigns, helps journalists track public sentiment, and helps researchers study collective attention.

## Why This Analysis Matters

- **Marketing insight:** Understanding tweet patterns helps brands optimize social media strategies during major events
- **Public sentiment:** Gauging public reaction to the first pandemic-era Olympics
- **Text analytics practice:** Real-world application of text processing and frequency analysis
- **Event analysis:** Demonstrates how to analyze temporal event data

## Dataset Overview

The dataset contains tweets related to the Tokyo 2020 Olympics.

**Expected columns may include:** tweet text, timestamp, user information, retweet count, like count, hashtags, language, and source.

## Dataset Source and License Notes

- **Source:** [Kaggle — Tokyo Olympics 2020 Tweets](https://www.kaggle.com/datasets/gpreda/tokyo-olympics-2020-tweets)
- **License:** Please check Kaggle for the latest license terms
- **Note:** Twitter data usage is subject to Twitter's Terms of Service

## Environment Setup

In [ ]:
!pip install -q pandas numpy matplotlib seaborn wordcloud kaggle

## Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re
import warnings
warnings.filterwarnings('ignore')

try:
    from wordcloud import WordCloud
    HAS_WORDCLOUD = True
except ImportError:
    HAS_WORDCLOUD = False

print("All imports loaded.")

## Configuration / Constants

In [ ]:
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

DATA_DIR = 'data'

## Dataset Download

Download from Kaggle using the API. Ensure `kaggle.json` is configured.

In [ ]:
import os
os.makedirs(DATA_DIR, exist_ok=True)

!kaggle datasets download -d gpreda/tokyo-olympics-2020-tweets -p {DATA_DIR} --unzip --force

print("Downloaded files:")
for f in os.listdir(DATA_DIR):
    if os.path.isfile(os.path.join(DATA_DIR, f)):
        size = os.path.getsize(os.path.join(DATA_DIR, f))
        print(f"  {f} ({size:,} bytes)")

## Data Loading

Load the tweets dataset and examine its structure.

In [ ]:
# Find CSV files in data directory
csv_files = [f for f in os.listdir(DATA_DIR) if f.endswith('.csv')]
print(f"CSV files found: {csv_files}")

# Load the main file
df = pd.read_csv(os.path.join(DATA_DIR, csv_files[0]))
print(f"\nDataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\nColumns: {list(df.columns)}")
df.head()

## Data Validation Checks

In [ ]:
print("=== Data Types ===")
print(df.dtypes)

print("\n=== Missing Values ===")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
missing_info = pd.DataFrame({'Count': missing, '%': missing_pct})
print(missing_info[missing_info['Count'] > 0])

print(f"\n=== Duplicates ===")
print(f"Duplicate rows: {df.duplicated().sum()}")

print("\n=== Sample data ===")
df.info()

## Data Cleaning

Parse dates, handle missing values, and prepare text features.

In [ ]:
# Identify the date column (common names in tweet datasets)
date_cols = [c for c in df.columns if any(d in c.lower() for d in ['date', 'time', 'created'])]
print(f"Date-like columns: {date_cols}")

# Parse the first date column found
if date_cols:
    date_col = date_cols[0]
    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
    df['date'] = df[date_col].dt.date
    df['hour'] = df[date_col].dt.hour
    df['day_of_week'] = df[date_col].dt.day_name()
    print(f"\nDate range: {df[date_col].min()} to {df[date_col].max()}")

# Identify text column
text_cols = [c for c in df.columns if any(t in c.lower() for t in ['text', 'tweet', 'content', 'body'])]
print(f"Text-like columns: {text_cols}")
if text_cols:
    text_col = text_cols[0]
    df[text_col] = df[text_col].fillna('')
    df['tweet_length'] = df[text_col].str.len()
    df['word_count'] = df[text_col].str.split().str.len().fillna(0).astype(int)

# Drop duplicates
initial_len = len(df)
df = df.drop_duplicates()
print(f"\nRemoved {initial_len - len(df)} duplicate rows")
print(f"Final shape: {df.shape}")

## Exploratory Data Analysis

### Tweet Volume Over Time

In [ ]:
if 'date' in df.columns:
    daily_counts = df.groupby('date').size()
    
    fig, ax = plt.subplots(figsize=(14, 6))
    daily_counts.plot(kind='line', ax=ax, marker='o', markersize=4, linewidth=2, color='#3498db')
    ax.fill_between(daily_counts.index, daily_counts.values, alpha=0.2, color='#3498db')
    ax.set_title('Daily Tweet Volume — Tokyo 2020 Olympics')
    ax.set_xlabel('Date')
    ax.set_ylabel('Number of Tweets')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    
    print(f"Peak day: {daily_counts.idxmax()} ({daily_counts.max():,} tweets)")
    print(f"Lowest day: {daily_counts.idxmin()} ({daily_counts.min():,} tweets)")
    print(f"Average daily tweets: {daily_counts.mean():,.0f}")
else:
    print("No date column found — skipping temporal analysis")

## Univariate Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Tweet length distribution
if 'tweet_length' in df.columns:
    axes[0].hist(df['tweet_length'], bins=50, edgecolor='black', alpha=0.7, color='#2ecc71')
    axes[0].set_title('Tweet Length Distribution')
    axes[0].set_xlabel('Characters')

# Word count distribution
if 'word_count' in df.columns:
    axes[1].hist(df['word_count'], bins=30, edgecolor='black', alpha=0.7, color='#e74c3c')
    axes[1].set_title('Word Count Distribution')
    axes[1].set_xlabel('Words')

# Hourly distribution
if 'hour' in df.columns:
    hour_counts = df['hour'].value_counts().sort_index()
    axes[2].bar(hour_counts.index, hour_counts.values, color='#9b59b6', alpha=0.8)
    axes[2].set_title('Tweet Distribution by Hour (UTC)')
    axes[2].set_xlabel('Hour')
    axes[2].set_ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# Language distribution (if available)
lang_cols = [c for c in df.columns if 'lang' in c.lower()]
if lang_cols:
    lang_col = lang_cols[0]
    top_langs = df[lang_col].value_counts().head(10)
    
    fig, ax = plt.subplots(figsize=(10, 5))
    top_langs.plot(kind='bar', ax=ax, color='#3498db')
    ax.set_title('Top 10 Tweet Languages')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("No language column found.")

## Bivariate / Multivariate Analysis

In [ ]:
# Day of week patterns
if 'day_of_week' in df.columns:
    dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    dow_counts = df['day_of_week'].value_counts().reindex(dow_order)
    
    fig, ax = plt.subplots(figsize=(10, 5))
    dow_counts.plot(kind='bar', ax=ax, color='#e67e22')
    ax.set_title('Tweets by Day of Week')
    ax.set_ylabel('Tweet Count')
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
# Engagement analysis (retweets, likes if available)
engagement_cols = [c for c in df.columns if any(e in c.lower() for e in ['retweet', 'like', 'favorite', 'reply'])]
print(f"Engagement columns found: {engagement_cols}")

if engagement_cols:
    fig, axes = plt.subplots(1, len(engagement_cols[:3]), figsize=(6*min(len(engagement_cols), 3), 5))
    if len(engagement_cols) == 1:
        axes = [axes]
    for i, col in enumerate(engagement_cols[:3]):
        df[col] = pd.to_numeric(df[col], errors='coerce')
        axes[i].hist(df[col].dropna(), bins=50, edgecolor='black', alpha=0.7)
        axes[i].set_title(f'{col} Distribution')
        axes[i].set_yscale('log')
        print(f"{col}: mean={df[col].mean():.1f}, median={df[col].median():.0f}, max={df[col].max():.0f}")
    plt.tight_layout()
    plt.show()
else:
    print("No engagement columns found in dataset.")

## Feature-Specific Insights

### Hashtag Analysis

In [ ]:
if text_cols:
    # Extract hashtags
    all_hashtags = []
    for text in df[text_col].dropna():
        tags = re.findall(r'#(\w+)', str(text).lower())
        all_hashtags.extend(tags)
    
    hashtag_counts = Counter(all_hashtags)
    top_20_tags = hashtag_counts.most_common(20)
    
    print("Top 20 Hashtags:")
    for tag, count in top_20_tags:
        print(f"  #{tag}: {count:,}")
    
    # Plot
    fig, ax = plt.subplots(figsize=(12, 6))
    tags, counts = zip(*top_20_tags)
    ax.barh(range(len(tags)), counts, color='#3498db')
    ax.set_yticks(range(len(tags)))
    ax.set_yticklabels([f'#{t}' for t in tags])
    ax.invert_yaxis()
    ax.set_title('Top 20 Hashtags')
    ax.set_xlabel('Frequency')
    plt.tight_layout()
    plt.show()
else:
    print("No text column found for hashtag extraction.")

In [ ]:
# Word cloud of tweet text
if HAS_WORDCLOUD and text_cols:
    all_text = ' '.join(df[text_col].dropna().astype(str))
    # Remove URLs and mentions
    all_text = re.sub(r'http\S+', '', all_text)
    all_text = re.sub(r'@\w+', '', all_text)
    all_text = re.sub(r'RT\s', '', all_text)
    
    wc = WordCloud(width=1200, height=600, background_color='white',
                   max_words=100, colormap='viridis').generate(all_text)
    
    fig, ax = plt.subplots(figsize=(14, 7))
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title('Word Cloud — Tokyo 2020 Tweets', fontsize=16)
    plt.tight_layout()
    plt.show()
else:
    print("WordCloud not available or no text column found.")

## Statistical Checks

Testing for patterns in tweet frequency.

In [ ]:
from scipy import stats

if 'hour' in df.columns:
    # Chi-squared test: are tweets uniformly distributed across hours?
    hour_obs = df['hour'].value_counts().sort_index().values
    expected = np.full_like(hour_obs, fill_value=len(df) / 24, dtype=float)
    chi2, p = stats.chisquare(hour_obs, expected)
    print(f"Hourly uniformity test: χ²={chi2:.2f}, p={p:.2e}")
    print(f"Tweets are {'NOT' if p < 0.05 else ''} uniformly distributed across hours.")

if 'day_of_week' in df.columns:
    # Chi-squared test: uniform across days of week?
    dow_obs = df['day_of_week'].value_counts().values
    expected_dow = np.full_like(dow_obs, fill_value=len(df) / 7, dtype=float)
    chi2, p = stats.chisquare(dow_obs, expected_dow)
    print(f"\nDay-of-week uniformity test: χ²={chi2:.2f}, p={p:.2e}")
    print(f"Tweets are {'NOT' if p < 0.05 else ''} uniformly distributed across days.")

if 'tweet_length' in df.columns:
    # Normality test on tweet length
    sample = df['tweet_length'].sample(min(5000, len(df)), random_state=42)
    stat, p = stats.shapiro(sample)
    print(f"\nTweet length normality (Shapiro-Wilk): W={stat:.4f}, p={p:.2e}")
    print(f"Tweet length is {'NOT normal' if p < 0.05 else 'approximately normal'}")

## Visual Analysis

### Heatmap of tweet activity by day and hour

In [ ]:
if 'date' in df.columns and 'hour' in df.columns:
    heatmap_data = df.groupby([df['date'], 'hour']).size().unstack(fill_value=0)
    
    fig, ax = plt.subplots(figsize=(16, 8))
    sns.heatmap(heatmap_data, cmap='YlOrRd', ax=ax)
    ax.set_title('Tweet Volume Heatmap: Date × Hour')
    ax.set_xlabel('Hour (UTC)')
    ax.set_ylabel('Date')
    plt.tight_layout()
    plt.show()
else:
    print("Date/hour columns not available for heatmap.")

In [ ]:
# Mention analysis
if text_cols:
    all_mentions = []
    for text in df[text_col].dropna():
        mentions = re.findall(r'@(\w+)', str(text))
        all_mentions.extend(mentions)
    
    mention_counts = Counter(all_mentions)
    top_mentions = mention_counts.most_common(15)
    
    if top_mentions:
        fig, ax = plt.subplots(figsize=(12, 6))
        names, counts = zip(*top_mentions)
        ax.barh(range(len(names)), counts, color='#e74c3c')
        ax.set_yticks(range(len(names)))
        ax.set_yticklabels([f'@{n}' for n in names])
        ax.invert_yaxis()
        ax.set_title('Top 15 Mentioned Accounts')
        ax.set_xlabel('Frequency')
        plt.tight_layout()
        plt.show()

## Key Findings

Key observations from the analysis (findings based on data examined):

1. **Tweet volume varies significantly by date** — Spikes likely correspond to major events (opening/closing ceremonies, key medal events)
2. **Hourly patterns reflect global time zones** — Peak hours indicate where most active audiences are located
3. **Olympic-related hashtags dominate** — Official and event-specific tags are most common
4. **English dominates** but the dataset shows global participation
5. **Tweet frequency is non-uniform** — Both daily and hourly patterns show significant variation from uniform distribution
6. **Engagement is highly skewed** — Most tweets get minimal engagement; a few go viral

## Limitations

- **Sample bias:** Kaggle datasets typically contain a subset of all tweets, not the complete Twitter firehose
- **Time coverage:** May not span the entire duration of the Olympics
- **No sentiment labels:** We analyze frequency and content, not sentiment (would require NLP models)
- **Bot activity:** Some tweets may come from automated accounts
- **Missing demographics:** No verified user demographics beyond language/location

## Recommendations / Next Steps

1. **Sentiment analysis:** Apply NLP models (VADER, TextBlob, or transformers) to classify tweet sentiment
2. **Topic modeling:** Use LDA or BERTopic to discover discussion themes
3. **Network analysis:** Map retweet and mention networks to find influencers
4. **Event correlation:** Align tweet spikes with specific Olympic events
5. **Bot detection:** Apply heuristics to identify and filter automated accounts

### How to Extend This Analysis

- Compare tweet patterns across multiple Olympic Games datasets
- Build a real-time tweet monitoring dashboard for future events
- Train a classifier to categorize tweets by sport or event type

## Common Mistakes

1. **Not handling timezones** — Tweets may be in UTC; failing to account for this distorts hourly patterns
2. **Counting retweets as original content** — Retweets inflate content frequency; filter or flag them
3. **Case-sensitive hashtag counting** — Always lowercase hashtags before counting
4. **Ignoring bot/spam tweets** — Can significantly skew frequency and content analysis
5. **Not cleaning URLs from text** — URLs add noise to word clouds and text analysis
6. **Treating tweet count as popularity** — High volume doesn't necessarily mean positive engagement

## Mini Challenge / Exercises

1. **Retweet ratio:** Calculate what percentage of tweets are retweets (start with 'RT'). How does this vary by day?

2. **URL usage:** What fraction of tweets contain URLs? Do tweets with URLs get more engagement?

3. **Hashtag co-occurrence:** Find the top 10 most common hashtag pairs that appear together in tweets.

4. **Peak hour analysis:** Identify the peak tweeting hour for each day. Is there a consistent pattern?

5. **Language diversity:** Calculate the Shannon entropy of the language distribution. What does it tell you about the global reach of the event?

In [ ]:
# Exercise space
# Exercise 1: Retweet ratio
# if text_cols:
#     df['is_retweet'] = df[text_col].str.startswith('RT ').fillna(False)
#     print(f"Retweet ratio: {df['is_retweet'].mean():.1%}")

print("Uncomment the exercises above and try them!")

## Final Summary / Key Takeaways

| Aspect | Finding |
|--------|--------|
| Volume | Highly variable — correlates with event schedule |
| Temporal | Non-uniform hourly/daily distribution |
| Content | Olympic hashtags dominate |
| Language | Multi-lingual but English-dominant |
| Engagement | Heavily right-skewed (most tweets get low engagement) |

Social media data around major events shows clear temporal patterns driven by the event schedule and global time zones. The Tokyo 2020 Olympics generated significant Twitter activity, providing valuable data for studying collective attention, content patterns, and social media dynamics during large-scale events.